# Paragraph citations · English and Finnish

Each paragraph may have several citations. Qwen selects up to eight source spans; Python copies them exactly. Paragraph-wide support is requested, not guaranteed. English keeps the frozen passages. Finnish uses a real Finnish query, Finnish HyDE, Finnish-only S retrieval and Finnish output. No model training.


In [ ]:
SETTINGS = {'source_report': 'reports/reranker-comparison-20261114T104507Z.json', 'finnish_question': 'Milloin talvisota käytiin, mistä se johtui ja miten se päättyi?', 'search_top_k': 30, 'citation_unit': 'paragraph', 'attribution_mode': 'source_spans', 'answer_language': 'English', 'context_tokens': 8192, 'target_words': 500, 'article_tokens': 4000, 'article_seconds': 300, 'reference_top_k': 3, 'reference_pair_tokens': 1024, 'attribution_batch_size': 2, 'max_spans_per_unit': 8, 'attribution_tokens': 1200, 'attribution_seconds': 90}


In [ ]:
from pathlib import Path
import sys, json, importlib, html
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'wikilens').is_dir())
sys.path.insert(0,str(ROOT))
from wikilens import citation_pipelines, paragraph_citations
importlib.reload(citation_pipelines); importlib.reload(paragraph_citations)
base = json.loads((ROOT / SETTINGS['source_report']).read_text())
examples = {}


In [ ]:
english = citation_pipelines.run_pipeline(base, 'post_hoc', SETTINGS)
examples['English'] = {'result':english,'preparation_seconds':0,'input_mode':'frozen_seven_passages'}
print(english['mechanical_metrics'])


In [ ]:
finnish_capture = paragraph_citations.prepare_finnish(base, SETTINGS)
fi_settings = {**SETTINGS, 'answer_language':'Finnish'}
finnish = citation_pipelines.run_pipeline(finnish_capture, 'post_hoc', fi_settings)
examples['Finnish'] = {'result':finnish,'preparation_seconds':finnish_capture['variants']['cross_encoder']['run']['preparation_seconds'], 'capture':finnish_capture}
print(finnish['mechanical_metrics'])


In [ ]:
from pathlib import Path
import json, html
from IPython.display import display, HTML
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'wikilens').is_dir())
p = ROOT / 'reports/paragraph-citations-latest.json'
if p.exists():
    saved = json.loads(p.read_text())
    for name, item in saved['examples'].items():
        print(name, item['result']['mechanical_metrics'])
        for paragraph in item['result']['sentences']:
            display(HTML('<h3>'+html.escape(paragraph['section'])+' · '+paragraph['claim_id']+'</h3><p>'+html.escape(paragraph['text'])+'</p>'))
            for citation in paragraph['citations']:
                display(HTML('<a href="'+html.escape(citation['revision_url'],quote=True)+'">['+citation['source_id']+']</a><blockquote>'+html.escape(citation['quote'])+'</blockquote>'))
            if not paragraph['citations']: print('[citation needed]')
else: print('Run scripts/run-paragraph-citations.py to save both examples.')
